# Fiche 3 — Clés étrangères, types et intégrité

| | |
|---|---|
| 🎯 **Objectif** | Choisir le bon type pour chaque colonne et faire respecter les liens entre tables |
| ⏱️ **Durée** | 45 min |
| 🧰 **Prérequis** | Fiches 1 et 2 |
| 🔗 **Dans le projet** | Parties 1 et 2 (schéma, création de la base) |

> **L'idée clé.** Types et contraintes sont des **garde-fous** : bien choisis, la base refuse les données absurdes ; mal choisis, elle les déforme **sans prévenir**.

## 1. ⚠️ Un code n'est pas un nombre

In [1]:
import sqlite3

import pandas as pd

# Une base « jetable » en mémoire : on peut tout casser, rien n'est écrit sur le disque.
# isolation_level=None : chaque instruction est validée aussitôt, comme dans SQLite Studio.
base = sqlite3.connect(":memory:", isolation_level=None)


def sql(requete: str) -> pd.DataFrame | None:
    """Exécute du SQL (une ou plusieurs instructions) et affiche le résultat s'il y en a un."""
    if requete.strip().rstrip(";").count(";"):
        base.executescript(requete)
        return None
    curseur = base.execute(requete)
    if curseur.description:
        return pd.DataFrame(curseur.fetchall(), columns=[c[0] for c in curseur.description])
    return None

sql("""
CREATE TABLE essai_types (code_departement INTEGER);
INSERT INTO essai_types VALUES ('01'), ('2A'), ('971');
""")
sql("SELECT code_departement, typeof(code_departement) AS type_stocke FROM essai_types")

,code_departement,type_stocke
0,1,integer
1,2A,text
2,971,integer


À coller dans SQLite Studio :
```sql
CREATE TABLE essai_types (code_departement INTEGER);
INSERT INTO essai_types VALUES ('01'), ('2A'), ('971');
SELECT code_departement, typeof(code_departement) AS type_stocke FROM essai_types;
```

SQLite n'a **rien refusé** : « 01 » est devenu l'entier 1 (le zéro a disparu), « 2A » est resté du texte. La
colonne mélange désormais deux types, et la clé commune reconstruite (« 01 » + « 103 ») ne correspondra plus à
celle des autres fichiers : les jointures perdront des lignes en silence. Le fichier DVF a déjà subi ce traitement
sur le code postal (`1170` au lieu de `01170`).

**Règle** : tout ce qui est un code (département, commune, code postal, région) est du **texte**.

## 2. Clé étrangère : un lien que la base vérifie (si on le lui demande)

In [2]:
sql("""
CREATE TABLE commune (id_codedep_codecommune TEXT PRIMARY KEY, nom_commune TEXT NOT NULL);
CREATE TABLE bien (
    id_bien                INTEGER PRIMARY KEY,
    id_codedep_codecommune TEXT NOT NULL REFERENCES commune (id_codedep_codecommune)
);
INSERT INTO commune VALUES ('34172', 'Montpellier');
""")
sql("INSERT INTO bien VALUES (1, '99999')")  # commune inconnue... et pourtant accepté !
sql("SELECT * FROM bien")

,id_bien,id_codedep_codecommune
0,1,99999


⚠️ **Piège propre à SQLite** : les clés étrangères sont **désactivées par défaut**. Il faut les activer à chaque
connexion, par une instruction exécutée **seule**, avant toute modification :

À coller dans SQLite Studio :
```sql
PRAGMA foreign_keys = ON;
```

In [3]:
sql("DELETE FROM bien")
sql("PRAGMA foreign_keys = ON")
try:
    sql("INSERT INTO bien VALUES (1, '99999')")
except sqlite3.IntegrityError as erreur:
    print("Maintenant SQLite refuse :", erreur)
sql("INSERT INTO bien VALUES (2, '34172')")  # commune existante : accepté

Maintenant SQLite refuse : FOREIGN KEY constraint failed


## 3. Pourquoi pas le code postal comme clé de la commune ?

Dans ton tableur, filtre le fichier des valeurs foncières sur le code postal **5100** (le zéro a disparu) : tu
verras Briançon, Montgenèvre et Névache. Filtre ensuite la commune « NICE » : tu verras 4 codes postaux différents.

- **Un code postal → plusieurs communes** : il n'est pas unique, ce n'est pas une clé.
- **Une commune → plusieurs codes postaux** : si on le range dans `commune`, il faut en choisir un et perdre les
  autres.

| Où ranger le code postal ? | Avantage | Coût |
|---|---|---|
| dans `commune` (comme l'ébauche fournie) | schéma plus simple | un seul code par commune : information perdue pour 57 communes |
| dans `bien` (adresse) | chaque adresse garde son vrai code | une colonne de plus dans une grande table |

Les deux se défendent : l'important est de **savoir justifier** son choix.

## 4. Les autres contraintes : NOT NULL, UNIQUE, CHECK

In [4]:
sql("""
CREATE TABLE livre (
    id_livre INTEGER PRIMARY KEY,
    isbn     TEXT NOT NULL UNIQUE,                                         -- obligatoire et sans doublon
    statut   TEXT NOT NULL CHECK (statut IN ('disponible', 'emprunté', 'perdu'))
);
""")
for requete in ["INSERT INTO livre VALUES (1, '978-2-07-036822-8', 'disponible')",
                "INSERT INTO livre VALUES (2, '978-2-07-036822-8', 'disponible')",   # isbn en double
                "INSERT INTO livre VALUES (3, '978-2-01-000000-0', 'vendu')"]:     # statut interdit
    try:
        sql(requete); print("accepté :", requete[26:])
    except sqlite3.IntegrityError as erreur:
        print("refusé  :", requete[26:], "→", erreur)

accepté : 1, '978-2-07-036822-8', 'disponible')
refusé  : 2, '978-2-07-036822-8', 'disponible') → UNIQUE constraint failed: livre.isbn
refusé  : 3, '978-2-01-000000-0', 'vendu') → CHECK constraint failed: statut IN ('disponible', 'emprunté', 'perdu')


### ✍️ À toi de jouer

Écris le `CREATE TABLE` d'une table `adherent` (bibliothèque) avec : un identifiant numéroté, un code postal qui doit faire exactement 5 caractères et garder son zéro initial, et une catégorie obligatoire qui ne peut valoir que 'enfant', 'adulte' ou 'senior'. Teste qu'un code postal '1170' est refusé.

In [5]:
# Ta réponse ici : sql(""" ... """)

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```sql
CREATE TABLE adherent (
    id_adherent INTEGER PRIMARY KEY,
    code_postal TEXT CHECK (length(code_postal) = 5),
    categorie   TEXT NOT NULL CHECK (categorie IN ('enfant', 'adulte', 'senior'))
);
INSERT INTO adherent VALUES (1, '1170', 'adulte');   -- refusé : CHECK constraint failed
```

`TEXT` garde le zéro ; `CHECK (length(…) = 5)` refuse un code abîmé au lieu de le stocker en silence.
</details>

## ✅ À retenir

1. Un code est du **texte** : en `INTEGER`, SQLite ne refuse rien mais abîme les codes (zéros perdus, types mélangés).
2. Une clé étrangère garantit qu'un lien pointe vers une ligne qui existe.
3. Dans SQLite, `PRAGMA foreign_keys = ON;` **seul, à chaque connexion**, sinon les FK sont ignorées.
4. `NOT NULL`, `UNIQUE`, `CHECK` : la base refuse elle-même les données absurdes.
5. Une clé doit être unique : le code postal ne l'est pas, le couple département + commune l'est.